# GPT-SoVITS WebUI

## Env Setup (Run Once Only)
## 环境配置, 只需运行一次

### 1.

In [1]:
%%writefile /content/setup.sh
set -e

cd /content

git clone https://github.com/RVC-Boss/GPT-SoVITS.git

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS

pip install ipykernel

bash install.sh --device CU126 --source HF --download-uvr5

Writing /content/setup.sh


### 2.

In [2]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:02:01
🔁 Restarting kernel...
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5930, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 5930 (delta 50), reused 16 (delta 16), pack-reused 5848 (from 4)
Receiving objects: 100% (5930/5930), 15.13 MiB | 17.87 MiB/s, done.
Resolving deltas: 100% (3380/3380), done.
Channels:
 - defaults
Platform: linux-64
Solving environment: / - done

## Package Plan ##

  environment location: /usr/local/envs/GPTSoVITS

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.5.14  |       h06a430

## データセット準備（Google Drive からダウンロード）

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, glob, os
DATA = '/content/dataset'
SRC = '/content/drive/MyDrive/99_未分類/gpt_sovits_data_20260703'
os.makedirs(DATA, exist_ok=True)
for z in glob.glob(f'{SRC}/*.zip'):
    zipfile.ZipFile(z).extractall(DATA)

# train.list の相対パスを絶対パスへ書き換え
for lst in glob.glob(f'{DATA}/*/train.list'):
    root = os.path.dirname(lst)
    rows = open(lst, encoding='utf-8').read().splitlines()
    rows = [f'{root}/' + r if r.startswith('wavs/') else r for r in rows]
    open(lst.replace('train.list', 'train_abs.list'), 'w', encoding='utf-8').write('\n'.join(rows) + '\n')
print('done:', glob.glob(f'{DATA}/*/train_abs.list'))

Mounted at /content/drive
done: ['/content/dataset/horikita/train_abs.list', '/content/dataset/chabashira/train_abs.list', '/content/dataset/kushida/train_abs.list', '/content/dataset/ayanokoji/train_abs.list']


## Launch WebUI
## 启动 WebUI

### starlette バージョン固定（gradio互換性エラー対策）
GPT-SoVITS の `webui.py` は `GPTSoVITS` という専用の conda 環境（python3.10）の中で動く。
ここで `source activate GPTSoVITS &&` を付けずに `pip install` すると、base の Colab カーネル（python3.12）に
インストールされてしまい、実際にサーバーが読み込む starlette には反映されない。

根本原因: gradio の `routes.py` が旧式の `templates.TemplateResponse(name, context)` という
位置引数の呼び出しをしており、新しい Starlette（0.46 以降）ではこの呼び出し方がサポートされず
`TypeError: unhashable type: 'dict'` がページ読み込みのたびに発生する。
参考: https://github.com/RVC-Boss/GPT-SoVITS/issues/2762

In [3]:
!source activate GPTSoVITS && pip install "starlette>=0.40.0,<0.46.0"

  Attempting uninstall: starlette
    Found existing installation: starlette 1.3.1
    Uninstalling starlette-1.3.1:
      Successfully uninstalled starlette-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.139.0 requires starlette>=0.46.0, but you have starlette 0.45.3 which is incompatible.


### ffmpeg / torchaudio ライブラリ不整合の修正
install.sh は `ffmpeg` と `torchaudio` をバージョン固定せずインストールするため、
実行タイミングによって以下が起きる（2026-07-03 のセッションで実際に両方発生）:

1. `ffmpeg: error while loading shared libraries: libjxl.so.0.11`
   → env の ffmpeg（conda-forge ビルド）は libjxl 0.11 の soname にリンクされているが、
   env に libjxl 0.11 が無い。**conda-forge の最新 libjxl は 0.12.0 なので、バージョン
   無指定で入れると `libjxl.so.0.12` になり直らない。`libjxl=0.11` の明示ピンが必須**
   （0.11 系は 0.11.0/0.11.1/0.11.2 が conda-forge に存在することを確認済み）。
2. `OSError: libcudart.so.13` / `Could not load ... _torchaudio.abi3.so`
   → torch は install.sh が cu126 で固定するが、torchaudio は PyPI から CUDA 13 ビルド
   （デフォルトバリアント）が入り ABI 不整合。cu126 インデックスに
   torch/torchaudio/torchcodec 2.9.1+cu126 / 0.9.1+cu126 が揃っていることを確認済み
   なので、3つをまとめて cu126 から再インストールして整合させる。


In [ ]:
!source activate GPTSoVITS && conda install --yes -c conda-forge "libjxl=0.11"
!source activate GPTSoVITS && pip install --force-reinstall torch torchaudio torchcodec --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://9abd4e48f5c78e948d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
